In [45]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.dates as mdates
from pandas.plotting import scatter_matrix
import seaborn as sns
from xgboost import XGBRegressor

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

from statsmodels.graphics.tsaplots import plot_acf

In [37]:
# Load Train dataset.
TRAIN_LOAD_LOCATION = '../data/raw/train.xlsx'

df = pd.read_excel(TRAIN_LOAD_LOCATION)

In [38]:
def df_transformer(df):

    YEARS_DICT = {1:2020, 2:2021, 3:2022}

    TEMP_COLS = [f"Site-{i + 1} Temp" for i in range(5)]
    GHI_COLS = [f"Site-{i + 1} GHI" for i in range(5)]

    # Holidays considered: New_Years_Day, Independence_Day, Thanksgiving_Day, Day_After_Thanksgiving, Christmas_Eve, Christmas_Day, New_Years_Eve
    list_of_notable_days = ["2020-01-01", "2020-07-04", "2020-11-26", "2020-11-27", "2020-12-24", "2020-12-25", "2020-12-31", 
                        "2021-01-01", "2021-07-04", "2021-11-25", "2021-11-26", "2021-12-24", "2021-12-25", "2021-12-31",
                        "2023-01-01", "2023-07-04", "2023-11-23", "2023-11-24", "2023-12-24", "2023-12-25", "2023-12-31"]

    base_t = 60

    notable_days = pd.to_datetime(list_of_notable_days)
    
    df_transformed = df.copy()
    
    # Transform temperature readings to Farenheit.
    df_transformed[TEMP_COLS] = df_transformed[TEMP_COLS].apply(lambda x: x * 9/5 + 32)
    
    # Average the temperature and GHI measurements across all sites, to explore temperature effects simply.
    df_transformed["avg_region_temp"] = df_transformed[TEMP_COLS].mean(axis=1)
    df_transformed["avg_region_ghi"] = df_transformed[GHI_COLS].mean(axis=1)
    
    # Create timestamps.
    df_transformed['Year'] = df_transformed['Year'].map(YEARS_DICT)
    df_transformed['Hour'] = df_transformed['Hour'] - 1
    df_transformed['timestamp'] = pd.to_datetime(df_transformed[['Year', 'Month', 'Day', 'Hour']])

    df_transformed["Hour_sin"] = np.sin(2*np.pi*df_transformed["Hour"]/24)
    df_transformed["Hour_cos"] = np.cos(2*np.pi*df_transformed["Hour"]/24)
    
    df_transformed["Month_sin"] = np.sin(2*np.pi*df_transformed["Month"]/24)
    df_transformed["Month_cos"] = np.cos(2*np.pi*df_transformed["Month"]/24)

    df_transformed["CDH"] = (df_transformed["avg_region_temp"] - base_t).clip(lower=0)
    df_transformed["HDH"] = (base_t - df_transformed["avg_region_temp"]).clip(lower=0)

    # Average of the temperature for the last 3, 6, and 24 hours.
    df_transformed["temp_3h"] = df_transformed["avg_region_temp"].rolling(3).mean()
    df_transformed["temp_6h"] = df_transformed["avg_region_temp"].rolling(6).mean()
    df_transformed["temp_24h"] = df_transformed["avg_region_temp"].rolling(24).mean()
    
    # Aggregate of the Cooling Degree Hours for the last 3, 6, and 24 hours.
    df_transformed["CDH_3h"] = df_transformed["CDH"].rolling(3).sum()
    df_transformed["CDH_6h"] = df_transformed["CDH"].rolling(6).sum()
    df_transformed["CDH_24h"] = df_transformed["CDH"].rolling(24).sum()
    
    # Aggregate of the Heating Degree Hours for the last 3, 6, and 24 hours.
    df_transformed["HDH_3h"] = df_transformed["HDH"].rolling(3).sum()
    df_transformed["HDH_6h"] = df_transformed["HDH"].rolling(6).sum()
    df_transformed["HDH_24h"] = df_transformed["HDH"].rolling(24).sum()

    # Ensure chronological ordering.
    df_transformed = df_transformed.sort_values("timestamp")

    df_transformed["Load_lag_1h"] = df_transformed["Load"].shift(1)
    df_transformed["Load_lag_2h"] = df_transformed["Load"].shift(2)
    df_transformed["Load_lag_3h"] = df_transformed["Load"].shift(3)
    df_transformed["Load_lag_6h"] = df_transformed["Load"].shift(6)
    df_transformed["Load_lag_12h"] = df_transformed["Load"].shift(12)
    df_transformed["Load_lag_24h"] = df_transformed["Load"].shift(24)
    df_transformed["Load_lag_48h"] = df_transformed["Load"].shift(48)
    df_transformed["Load_lag_168h"] = df_transformed["Load"].shift(168)

    df_transformed = df_transformed.dropna()

    # Add feature indicating weekend.
    df_transformed['is_weekend'] = (df_transformed['timestamp'].dt.weekday >= 5).astype(int)

    # Add feature indicating notable days.
    df_transformed['is_notable_day'] = (df_transformed['timestamp'].dt.normalize().isin(notable_days)).astype(int)

    # Rearrange columns

    cols = list(df_transformed.columns)
    cols.remove("timestamp")
    df_transformed = df_transformed[["timestamp"] + cols]

    return df_transformed

In [41]:
df = df_transformer(df)

In [43]:
df.head()

,timestamp,Year,Month,Day,Hour,Load,Site-1 Temp,Site-2 Temp,Site-3 Temp,Site-4 Temp,...,Load_lag_1h,Load_lag_2h,Load_lag_3h,Load_lag_6h,Load_lag_12h,Load_lag_24h,Load_lag_48h,Load_lag_168h,is_weekend,is_notable_day
168,2020-01-08 00:00:00,2020,1,8,0,1956,46.22,48.02,48.02,48.56,...,2018.0,2176.0,2386.0,2708.0,1885.0,1902.0,1882.0,1997.0,0,0
169,2020-01-08 01:00:00,2020,1,8,1,1925,46.22,47.12,46.76,47.48,...,1956.0,2018.0,2176.0,2634.0,1952.0,1839.0,1824.0,1921.0,0,0
170,2020-01-08 02:00:00,2020,1,8,2,1899,48.02,49.64,45.32,47.66,...,1925.0,1956.0,2018.0,2529.0,2079.0,1801.0,1794.0,1861.0,0,0
171,2020-01-08 03:00:00,2020,1,8,3,1901,52.88,52.16,44.96,47.84,...,1899.0,1925.0,1956.0,2386.0,2244.0,1811.0,1810.0,1833.0,0,0
172,2020-01-08 04:00:00,2020,1,8,4,1969,51.44,52.88,44.24,46.40,...,1901.0,1899.0,1925.0,2176.0,2485.0,1900.0,1912.0,1847.0,0,0


In [73]:
splits_df_loc = "../data/splits/split_bounds.csv"
splits_df = pd.read_csv(splits_df_loc)

def linear_xgbr_rmse_on_features(df, features_to_train_lin, xgbr_depth, xgbr_estimators):

    #print("Now training on: ", features_to_train_lin)

    rmse_list = []

    for i in range(1, 6):
        split = f"split_{i}"
    
        row = splits_df[splits_df["split"] == split].iloc[0]
        train_start_date = row["train_start_date"]
        train_end_date = row["train_end_date"]
        val_start_date = row["val_start_date"]
        val_end_date = row["val_end_date"]
    
        train_mask = (df["timestamp"] >= train_start_date) & (df["timestamp"] < train_end_date)
        val_mask = (df["timestamp"] >= val_start_date) & (df["timestamp"] < val_end_date)
        
        train_split = df[train_mask].drop(columns=["timestamp"])
        val_split = df[val_mask].drop(columns=["timestamp"])
    
        X_train = train_split[features_to_train_lin]
        y_train = train_split["Load"]
    
        X_val = val_split[features_to_train_lin]
        y_val = val_split["Load"]
    
        model = LinearRegression()
        model.fit(X_train, y_train)
        
        residuals_train = y_train - model.predict(X_train)

        xgbr = XGBRegressor( objective='reg:squarederror', n_estimators = xgbr_estimators, learning_rate=0.05, max_depth = xgbr_depth, subsample=0.8)

        xgbr.fit(train_split, residuals_train)
    
        y_lin_pred = model.predict(X_val)
        y_xgbr_pred = xgbr.predict(val_split)
        rmse = root_mean_squared_error(y_val, y_lin_pred + y_xgbr_pred)
    
        rmse_list.append(rmse)
    print("RMSEs: ", rmse_list)
    print(f"Average RMSE across splits: {np.mean(rmse_list):.2f} ± {np.std(rmse_list):.2f}\n")

In [87]:
splits_df_loc = "../data/splits/split_bounds.csv"
splits_df = pd.read_csv(splits_df_loc)

def xgbr_rmse_on_features(df, features_to_train_lin, xgbr_depth, xgbr_estimators):

    #print("Now training on: ", features_to_train_lin)

    rmse_list = []

    for i in range(1, 6):
        split = f"split_{i}"
    
        row = splits_df[splits_df["split"] == split].iloc[0]
        train_start_date = row["train_start_date"]
        train_end_date = row["train_end_date"]
        val_start_date = row["val_start_date"]
        val_end_date = row["val_end_date"]
    
        train_mask = (df["timestamp"] >= train_start_date) & (df["timestamp"] < train_end_date)
        val_mask = (df["timestamp"] >= val_start_date) & (df["timestamp"] < val_end_date)
        
        train_split = df[train_mask].drop(columns=["timestamp"])
        val_split = df[val_mask].drop(columns=["timestamp"])
    
        X_train = train_split.drop(columns=["Load"])
        y_train = train_split["Load"]
    
        X_val = val_split.drop(columns=["Load"])
        y_val = val_split["Load"]
    
        model = XGBRegressor( objective='reg:squarederror', n_estimators = xgbr_estimators, learning_rate=0.1, max_depth = xgbr_depth, subsample=0.8)

        model.fit(X_train, y_train)
    
        y_pred = model.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
    
        rmse_list.append(rmse)
    print("RMSEs: ", rmse_list)
    print(f"Average RMSE across splits: {np.mean(rmse_list):.2f} ± {np.std(rmse_list):.2f}\n")

In [95]:
for depth in range(5, 7):
    for estimators in range(50, 1050, 50):
        print(f"depth:{depth}, estimators:{estimators}")
        xgbr_rmse_on_features(df = df, 
                                    features_to_train_lin = ["Hour_sin", "Hour_cos", "Month", "Day", "temp_6h", "avg_region_ghi", "is_weekend", "is_notable_day", "CDH", "HDH", "Load_lag_1h", "Load_lag_2h", "Load_lag_3h", "Load_lag_24h"],
                                    xgbr_depth = depth,
                                    xgbr_estimators = estimators)



depth:5, estimators:50
RMSEs:  [69.94294301470987, 71.66975239626522, 69.63860439102629, 83.50058949680826, 78.09398968071244]
Average RMSE across splits: 74.57 ± 5.41

depth:5, estimators:100
RMSEs:  [52.660338273408826, 51.250918049078415, 49.84780131192801, 61.138654148496755, 65.64991163240215]
Average RMSE across splits: 56.11 ± 6.18

depth:5, estimators:150
RMSEs:  [51.25879282444091, 47.51169558982209, 47.07534520905264, 57.72920806865205, 63.650250329206244]
Average RMSE across splits: 53.45 ± 6.37

depth:5, estimators:200
RMSEs:  [51.14534537015911, 46.215506051009534, 45.84346493368005, 56.29859208127086, 62.66004671206256]
Average RMSE across splits: 52.43 ± 6.38

depth:5, estimators:250
RMSEs:  [51.171404655149495, 45.62261001519479, 45.20744360260727, 55.46169355080438, 62.03080312610166]
Average RMSE across splits: 51.90 ± 6.33

depth:5, estimators:300
RMSEs:  [51.347112902024726, 45.266375391809554, 44.78842294468482, 54.54619259016031, 61.30479658197726]
Average RMSE ac